# 01. Data Cleaning & Sensor Physical Bounds Auditing
## Water Quality Prediction via Gradient Boosting & Deep Neural Networks (Phase 1)

### Objectives:
- Enforce physical and chemical bounds on water telemetry (pH, Temperature, Dissolved Oxygen, BOD, Nutrients).
- Identify and audit sensor telemetry errors without silent data deletion.
- Implement the flag_and_clean_sensor_anomalies protocol (NaN substitution + boolean fault indicator columns).
- Investigate sensor detection limits (left-censored zeros in nutrient parameters).
- Compare pre-cleaning vs post-cleaning distributions.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.data.clean import audit_physical_bounds, flag_and_clean_sensor_anomalies, DEFAULT_PHYSICAL_BOUNDS

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline
print("Cleaning & auditing environment ready!")

### 1. Ingest Data Subset

In [ ]:
dataset_path = ROOT_DIR / "data" / "raw" / "Combined_dataset.csv"
df_raw = pd.read_csv(dataset_path, nrows=100000)
print(f"Loaded {len(df_raw):,} records for bounds diagnostics.")

### 2. Physical & Chemical Bounds Diagnostic Audit

In [ ]:
bounds_report = audit_physical_bounds(df_raw)
display(bounds_report)

### 3. Anomaly Analysis: Impossible pH and Extreme Temperatures

In [ ]:
# Inspect rows with impossible pH (> 14)
invalid_ph = df_raw[df_raw["pH (ph units)"] > 14.0]
print(f"Found {len(invalid_ph)} samples with pH > 14.0 (Max pH observed: {df_raw['pH (ph units)'].max():.2f})")
invalid_ph[["Country", "Area", "Waterbody Type", "pH (ph units)", "Temperature (cel)", "CCME_WQI"]].head()

In [ ]:
# Inspect rows with extreme water temperature (> 50 C)
invalid_temp = df_raw[df_raw["Temperature (cel)"] > 50.0]
print(f"Found {len(invalid_temp)} samples with Temperature > 50 C (Max Temp: {df_raw['Temperature (cel)'].max():.2f} C)")
invalid_temp[["Country", "Area", "Waterbody Type", "Temperature (cel)", "Dissolved Oxygen (mg/l)"]].head()

### 4. Execute flag_and_clean_sensor_anomalies Protocol
Rather than deleting entire rows or clipping values, we set impossible values to NaN and generate binary fault indicators ([sensor]_is_sensor_fault = 1).

In [ ]:
df_cleaned, anomaly_summary = flag_and_clean_sensor_anomalies(df_raw, action="nan_and_flag")
print("Anomaly counts flagged per column:", anomaly_summary)

fault_cols = [c for c in df_cleaned.columns if c.endswith("_is_sensor_fault")]
print(f"Created {len(fault_cols)} sensor fault indicator columns:", fault_cols)

### 5. Detection Limits (Left-Censored Zeros)
In chemical water analysis, concentrations below detection limits are typically recorded as 0.0.

In [ ]:
nutrient_cols = ["Ammonia (mg/l)", "Biochemical Oxygen Demand (mg/l)", "Orthophosphate (mg/l)", "Nitrate (mg/l)"]
censored_summary = []
for col in nutrient_cols:
    zero_count = (df_cleaned[col] == 0.0).sum()
    censored_summary.append({
        "Parameter": col,
        "Zero/Censored Count": zero_count,
        "Censored Percentage": round((zero_count / len(df_cleaned)) * 100, 2)
    })
pd.DataFrame(censored_summary)

### Summary of Data Cleaning Rigor:
- Enforced thermodynamic constraints on pH (0 to 14) and Temperature (-5 to 50 C).
- Flagged sensor faults without dropping records, preserving statistical power.
- Quantified left-censored detection limits for downstream tree splits and imputer handling.